# OpenHealth Price Transparency Analysis

This notebook demonstrates how to access healthcare pricing data using the **OpenHealthDP API** and perform temporal trend analysis to compare costs over time.

### Prerequisites
- Ensure the OpenHealthDP server is running (usually on `http://localhost:3001`)
- Install dependencies: `pip install requests pandas matplotlib seaborn`

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
BASE_URL = "http://localhost:3001/api/v1/transparency"
API_KEY = "AI4H2-PUBLIC-2023-BETA"  # Default beta key

def fetch_procedure_data(procedure_id, city=None):
    headers = {"x-api-key": API_KEY}
    params = {"procedureId": procedure_id}
    if city:
        params["city"] = city
        
    response = requests.get(BASE_URL, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()["data"]
        return pd.DataFrame(data)
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

## 1. Load Data for MRI Brain (CPT:70551)
Let's fetch pricing data for a common imaging procedure across all providers.

In [ ]:
procedure_id = "CPT:70551" # MRI Brain (without contrast)
df = fetch_procedure_data(procedure_id)

if df is not None:
    print(f"Loaded {len(df)} records for {procedure_id}")
    display(df.head())

## 2. Basic Statistical Summary
We can look at the variation between the **Gross Charge** (Sticker Price) and the **Allowed Amount** (Negotiated Rate).

In [ ]:
if df is not None:
    stats = df[['avg_charge', 'avg_allowed']].describe()
    display(stats)

## 3. Temporal Trend Analysis
Comparing the temporal trend of **Gross Charges** vs. **Allowed Amounts** over the years.

In [ ]:
if df is not None and 'source_year' in df.columns:
    # Group by year and calculate mean prices
    trend_df = df.groupby('source_year')[['avg_charge', 'avg_allowed']].mean().reset_index()
    
    plt.figure(figsize=(12, 6))
    
    sns.lineplot(data=trend_df, x='source_year', y='avg_charge', marker='o', label='Mean Gross Charge')
    sns.lineplot(data=trend_df, x='source_year', y='avg_allowed', marker='o', label='Mean Negotiated Amount')
    
    plt.title(f'Temporal Price Trends for {procedure_id}')
    plt.xlabel('Year')
    plt.ylabel('Average Price ($)')
    plt.xticks(trend_df['source_year'].unique())
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()
else:
    print("source_year column not found in data.")